# Notebook 09 - Kizomba Eval + Extended Set

Run the interactive timeline viz plus the Gemma `rhythm_anatomy` (Phase 36/37a),
`listening_guide` (Phase 33), `kizomba_tutor` coaching pass, and verified
`kizomba_drills` (Phase 37c) across selected kizomba source folders:

- `data/songs/eval_set/kizomba/` - the curated base eval set
- `data/songs/extended_set/kizomba/` - the broader stress-test set

Both sources are enabled by default. Flip `INCLUDE_EVAL_SET` or
`INCLUDE_EXTENDED_SET` in the config cell to run only one source.

Pattern is the same as notebook 07 (per-track: analyze -> timeline -> Gemma
coaching), but the track list is auto-discovered from the selected source roots
and each output heading is source-labeled so eval-set tracks do not get lost
inside the extended-set smoke.

Phase 36/37a reframe: `rhythm_anatomy` runs ONCE at the top to frame what
kizomba sounds like as a genre, including sub-style hints, before the per-track
loop. Phase 33 reframe: the per-track listening guide runs *before* the tutor for
every track so the demo flow on each song reads as orientation -> coaching,
mirroring notebook 00. Phase 37c adds a per-track drill smoke: raw Gemma drill
plans are preserved, but learner-facing drills are normalized against DSP phases
before review. Toggle with `RUN_RHYTHM_ANATOMY` / `RUN_LISTENING_GUIDE` /
`RUN_KIZOMBA_DRILLS` in the config cell.

Honest scope reminders (carried over from notebook 07):
* Kizomba downbeat / `"1"` detection is **out of scope** - every kizomba prompt forbids naming a `"1"` position.
* Where the analysis reports `beat: subtle`, the tutor warns the learner instead of pretending the pulse is locked.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

from IPython.display import HTML, Markdown, display

from rytmi import analyze, load_audio
from rytmi import viz as rytmi_viz
from rytmi.dsp import describe_transitions, extract_transitions
from rytmi.llm import explain_rhythm, generate, load_cloud_model, polish_kizomba_tutor_output
from rytmi.prompts import (
    QUESTION_KIZOMBA_DRILLS,
    QUESTION_KIZOMBA_TRANSITIONS,
    QUESTION_KIZOMBA_TUTOR,
    QUESTION_LISTENING_GUIDE,
    QUESTION_RHYTHM_ANATOMY,
    format_unified_timeline,
    verify_kizomba_drills_output,
    verify_kizomba_transitions_output,
)
from rytmi.vocal_activity import default_vocal_activity_source

# Local Gemma E4B via Ollama is the project default. Override with
# RYTMI_API_BASE_URL / RYTMI_API_KEY env vars (e.g. for a cloud endpoint).
CLOUD_BASE_URL = os.environ.get("RYTMI_API_BASE_URL", "http://localhost:11434/v1")
CLOUD_API_KEY = os.environ.get("RYTMI_API_KEY", "ollama")
CLOUD_MODEL_ID = os.environ.get("RYTMI_MODEL_ID", "google/gemma-4-26b-a4b-it")

EVAL_SET = Path("../data/songs/eval_set/kizomba")
EXTENDED_SET = Path("../data/songs/extended_set/kizomba")
AUDIO_EXTS = {".mp3", ".wav", ".flac", ".m4a", ".ogg"}

# Source toggles. Both default to True so notebook 09 covers the curated base
# eval set and the broader extended smoke set in one run.
INCLUDE_EVAL_SET = True
INCLUDE_EXTENDED_SET = True

SOURCE_ROOTS: list[tuple[str, Path]] = []
if INCLUDE_EVAL_SET:
    SOURCE_ROOTS.append(("eval_set", EVAL_SET))
if INCLUDE_EXTENDED_SET:
    SOURCE_ROOTS.append(("extended_set", EXTENDED_SET))


def _discover_tracks(root: Path) -> list[Path]:
    if not root.exists():
        print(f"missing source folder: {root}")
        return []
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in AUDIO_EXTS)


# Auto-discover every audio file under the enabled source roots. Override TRACKS
# below if you want to run on a subset; update TRACK_SOURCE too if source labels
# matter for your manual subset.
TRACKS: list[Path] = []
TRACK_SOURCE: dict[Path, str] = {}
for source_label, root in SOURCE_ROOTS:
    source_tracks = _discover_tracks(root)
    TRACKS.extend(source_tracks)
    for path in source_tracks:
        TRACK_SOURCE[path] = source_label

PIXELS_PER_SECOND = 80

# Phase 29b polish pass - runs a second LLM call per track that rewrites the
# one-pass draft against a stricter rubric. Off by default to keep the run
# fast; flip to True to compare side-by-side.
RUN_POLISH = True

# Phase 33 listening guide - runs a pre-tutor orientation pass per track
# (anatomy + difficulty map, prose). On by default since it's cheap relative
# to tutor + polish; flip to False to skip.
RUN_LISTENING_GUIDE = True

# Phase 36 genre intro - runs ONCE at the top, before the per-track loop.
# Frames what kizomba sounds like as a genre so each track's listening
# guide can focus on what's specific. Cheap (one LLM call total).
RUN_RHYTHM_ANATOMY = True

# Phase 37c drill verifier smoke - one extra LLM call per track. The raw
# Gemma draft is stored, but the printed/dumped practice plan is verified
# against DSP phases before learner-facing review.
RUN_KIZOMBA_DRILLS = True

# Phase 38 vocal-activity envelope - mirrors notebook 05's plumbing. When
# True, a Demucs-based vocal source separates vocals per track and feeds the
# envelope into analyze(...) so the section labeler can carve `instrumental`
# sub-sections out of vocal-quiet, high-energy runs (`_relabel_vocal_drop_
# instrumentals` in dsp.py). Without this, vocal-drop runs collapse into
# generic `main` labels and the per-track listening_guide / tutor lose the
# nuance. Flip to False on machines without Demucs / VRAM.
RUN_VOCAL_ACTIVITY = True

# Phase 40 transitions - one extra LLM call per track. extract_transitions
# walks the phase list and emits one Transition per boundary; the prompt
# writes anticipation + re-entry coaching; verify_kizomba_transitions_output
# drops invented boundaries and fills missing ones with template text.
RUN_KIZOMBA_TRANSITIONS = True

# Phase 40d - same-label transitions. When True, extract_transitions also
# emits same-label energy-shift boundaries (e.g. main x4 medium -> main x2
# high). When False, only label-change boundaries appear. Flip to False if
# the same-label transitions read as cruft vs the section-role vocabulary
# already in kizomba_tutor's per-phase coaching.
INCLUDE_SAME_LABEL_TRANSITIONS = True

# Phase 40e — RUN_TRANSITION_RETRY=True asks Gemma to write any missing
# transition line one more time before the deterministic _fallback_transition_tail
# substitution kicks in. Honest with the project's signature pattern
# (code identifies, Gemma writes, code verifies) and reduces template
# repetition across same-label T# fills. Costs one extra Gemma call per
# missing boundary; flip to False to skip.
RUN_TRANSITION_RETRY = True

# Phase 41-lite — INCLUDE_PHASE_FEATURES=True appends qualitative texture
# (bass-driven / percussive / balanced from harm/perc ratios) and onset-density
# (sparse / steady / busy from onsets-per-beat) tags to each phase summary line
# in the analysis dump fed to Gemma. Read-only DSP, no schema change. Goal:
# more song-specific tutor and transition coaching. Flip to False to A/B.
INCLUDE_PHASE_FEATURES = False  # Phase 41-lite trial: tried, mixed result; off by default. Flip to True to revisit.

# Phase 41-D — INCLUDE_PHASE_VOCAL=True appends a per-phase vocal-presence
# tag (present / sparse / quiet) to the analysis dump fed to Gemma. The
# vocal_ratio per section is populated by analyze() when a vocal_env (Demucs)
# is provided. Vocals come and go more than HPSS texture does within a
# kizomba song, so this is the candidate signal most likely to give Gemma
# real inter-phase contrast. A/B trial — flip to False to compare.
INCLUDE_PHASE_VOCAL = False  # Phase 41-D trial: tried, same null result as 41-lite. See 41-2026-05-11 note.

print(f"discovered {len(TRACKS)} kizomba tracks across {len(SOURCE_ROOTS)} enabled sources")
for source_label, root in SOURCE_ROOTS:
    source_tracks = [path for path in TRACKS if TRACK_SOURCE.get(path) == source_label]
    print(f"{source_label}: {len(source_tracks)} tracks under {root}")
    for path in source_tracks:
        print(f"  - {path.name}")

## Load the Gemma client (cheap — just an HTTP wrapper)

In [ ]:
processor, model = load_cloud_model(
    base_url=CLOUD_BASE_URL,
    api_key=CLOUD_API_KEY,
    model_id=CLOUD_MODEL_ID,
)

## Vocal-activity source (Phase 38)

Build a Demucs-based vocal source once so every per-track call can separate
vocals and feed an envelope into `analyze(...)`. Without this the section
labeler can't carve `instrumental` sub-sections out of vocal-quiet, high-energy
runs (see `_relabel_vocal_drop_instrumentals` in `src/rytmi/dsp.py`) — exactly
the behaviour notebook 05 has had since Phase 9. Notebook 09 was previously
running unaware of vocals and collapsing those runs into plain `main`. The
chained source falls back gracefully if Demucs is unavailable. Flip
`RUN_VOCAL_ACTIVITY = False` in the config cell to disable.

In [ ]:
vocal_source = None
if RUN_VOCAL_ACTIVITY:
    vocal_source = default_vocal_activity_source(prefer="demucs")
    print(f"[vocal-activity] configured source: {vocal_source!r}")
else:
    print("RUN_VOCAL_ACTIVITY=False; analyze() runs without vocal envelope "
          "(no `instrumental` sub-section labeling)")

## Genre intro - what kizomba sounds like, broadly (Phase 36/37a)

Before the per-track loop, run `QUESTION_RHYTHM_ANATOMY` once to frame what
kizomba sounds like as a genre. The output is a compact two-paragraph intro the
user reads first: rhythmic anatomy first, then sub-style hints. Every per-track
listening guide and tutor below adds the song-specific detail. We load the first
track just to satisfy the `explain_rhythm` API shape - the prompt is genre-level
and explicitly forbidden from referring to a specific track.

In [ ]:
RHYTHM_ANATOMY = ""
if RUN_RHYTHM_ANATOMY and TRACKS:
    _seed_audio = load_audio(TRACKS[0])
    _seed_vocal_env = None
    if vocal_source is not None:
        try:
            _seed_vocal_env = vocal_source.compute(_seed_audio)
        except Exception as _e:
            print(f"[vocal-activity] seed FAILED ({_e}); continuing without envelope")
    _seed_analysis = analyze(_seed_audio, dance_style="kizomba", vocal_env=_seed_vocal_env)
    RHYTHM_ANATOMY = explain_rhythm(_seed_analysis, QUESTION_RHYTHM_ANATOMY, processor, model)
    print(RHYTHM_ANATOMY)
else:
    print("RUN_RHYTHM_ANATOMY=False or TRACKS empty; skipping genre intro")

## Per-track helper: analyze, draw, ask Gemma

* **Orange bold lines** = downbeats (kizomba: out-of-scope, treat as visual aid only)
* **Green dashed lines** = beats, numbered within each measure
* **Red thin lines** = detected onsets
* **Red cursor** = playback position (click timeline to seek)

In [ ]:
# PER_TRACK accumulates everything so the dump cell at the bottom can write
# all per-track outputs to a single Markdown file without re-running anything.
PER_TRACK: dict[str, dict] = {}


def _format_kizomba_drill_stats(stats: dict) -> str:
    return " ".join(
        f"{key}={stats.get(key, 0)}"
        for key in (
            "parsed",
            "repaired_ranges",
            "duplicate_phases",
            "filled_missing",
            "skipped_lines",
            "output_lines",
        )
    )


def _format_kizomba_transitions_stats(stats: dict) -> str:
    return " ".join(
        f"{key}={stats.get(key, 0)}"
        for key in (
            "parsed",
            "boundaries_matched",
            "boundaries_invented",
            "boundaries_missing_filled",
            "skipped_lines",
            "output_lines",
            "retried",
            "retry_succeeded",
        )
        if key in stats
    )


def _track_base_name(path: Path) -> str:
    return path.stem.split(" [")[0]


def _track_source(path: Path) -> str:
    return TRACK_SOURCE.get(path, "manual")


def _track_display_name(path: Path) -> str:
    return f"{_track_source(path)}: {_track_base_name(path)}"


def tutor(path: Path) -> tuple[str, str]:
    """Run analyze + timeline + optional Gemma passes for the track path."""
    source = _track_source(path)
    name = _track_display_name(path)
    display(Markdown(f"---\n## {name}"))
    audio = load_audio(path)
    vocal_env = None
    if vocal_source is not None:
        try:
            vocal_env = vocal_source.compute(audio)
        except Exception as e:
            print(f"  vocal-activity FAILED ({e}); continuing without envelope")
    a = analyze(audio, dance_style="kizomba", vocal_env=vocal_env)
    header = (
        f"source={source} | file={path.name}\n"
        f"  tempo={a.tempo:.1f} BPM | beats={len(a.beats.times)} | "
        f"sections={len(a.sections)} | vocal_env={'yes' if vocal_env is not None else 'no'}"
    )
    print(header)
    section_lines = []
    for sec in a.sections:
        bc = sec.rhythm_features.beat_clarity if sec.rhythm_features else float('nan')
        line = (
            f"    {sec.label:8s} {sec.start_s:6.1f}s-{sec.end_s:6.1f}s  "
            f"beat_clarity={bc:.2f}"
        )
        print(line)
        section_lines.append(line)
    display(HTML(rytmi_viz.interactive_timeline(
        a, title=f"{name} - kizomba beats",
        pixels_per_second=PIXELS_PER_SECOND,
    )))
    listening = ""
    if RUN_LISTENING_GUIDE:
        print("\n--- Gemma listening_guide (pre-listen orientation + difficulty map) ---")
        listening = explain_rhythm(a, QUESTION_LISTENING_GUIDE, processor, model)
        print(listening)
    print("\n--- Gemma kizomba_tutor (one-pass) ---")
    draft = explain_rhythm(
    a, QUESTION_KIZOMBA_TUTOR, processor, model,
    include_phase_features=INCLUDE_PHASE_FEATURES,
    include_phase_vocal=INCLUDE_PHASE_VOCAL,
)
    print(draft)
    drills = ""
    drills_raw = ""
    drills_verified_stats = ""
    if RUN_KIZOMBA_DRILLS:
        print("\n--- Gemma kizomba_drills (verified Phase 37c smoke) ---")
        drills_raw = explain_rhythm(a, QUESTION_KIZOMBA_DRILLS, processor, model)
        drills_verified = verify_kizomba_drills_output(drills_raw, a.phases)
        drills = drills_verified.cleaned or drills_raw
        drills_verified_stats = _format_kizomba_drill_stats(drills_verified.stats)
        print(drills)
        print(f"\n[verifier: {drills_verified_stats}]")
    transitions_table = ""
    transitions_text = ""
    transitions_raw = ""
    transitions_verified_stats = ""
    if RUN_KIZOMBA_TRANSITIONS:
        print("\n--- Gemma kizomba_transitions (Phase 40 + 40d smoke) ---")
        transitions_list = extract_transitions(
            a.phases, include_same_label=INCLUDE_SAME_LABEL_TRANSITIONS,
        )
        transitions_table = describe_transitions(
            a, include_same_label=INCLUDE_SAME_LABEL_TRANSITIONS,
        )
        print(transitions_table)
        print()
        transitions_raw = explain_rhythm(
            a, QUESTION_KIZOMBA_TRANSITIONS, processor, model,
            include_same_label_transitions=INCLUDE_SAME_LABEL_TRANSITIONS,
                include_phase_features=INCLUDE_PHASE_FEATURES,
                include_phase_vocal=INCLUDE_PHASE_VOCAL,
        )
        transitions_verified = verify_kizomba_transitions_output(
            transitions_raw, transitions_list,
            retry_callback=(
                (lambda prompt: generate(processor, model, prompt, max_new_tokens=128))
                if RUN_TRANSITION_RETRY else None
            ),
        )
        transitions_text = transitions_verified.cleaned or transitions_raw
        transitions_verified_stats = _format_kizomba_transitions_stats(
            transitions_verified.stats,
        )
        print(transitions_text)
        print(f"\n[verifier: {transitions_verified_stats}]")
    PER_TRACK[name] = {
        "source": source,
        "file": path.name,
        "header": header.strip(),
        "sections_text": "\n".join(section_lines),
        "listening": listening,
        "draft": draft,
        "drills": drills,
        "drills_raw": drills_raw,
        "drills_verified_stats": drills_verified_stats,
        "transitions_table": transitions_table,
        "transitions": transitions_text,
        "transitions_raw": transitions_raw,
        "transitions_verified_stats": transitions_verified_stats,
        "polished": "",  # filled in by the polish cell if RUN_POLISH=True
    }
    return name, draft

## Run on every track in the enabled kizomba sources

Errors on individual tracks are caught so one bad file does not abort the
whole batch. The failure prints, the loop moves on, and the final count still
shows how many selected tracks completed.

In [ ]:
DRAFTS: dict[str, str] = {}
for p in TRACKS:
    if not p.exists():
        print(f"missing: {p}")
        continue
    try:
        name, draft = tutor(p)
        DRAFTS[name] = draft
    except Exception as e:
        print(f"  {p.name}: FAILED ({e})")
print(f"\nProcessed {len(DRAFTS)}/{len(TRACKS)} tracks")

## Optional: polished rewrite pass

Set `RUN_POLISH = True` in the config cell above to enable. Runs a second LLM
call per draft against a stricter coaching rubric — preserves every P# header,
time span, and beat tag, but tightens the coaching text.

In [ ]:
if RUN_POLISH:
    for name, draft in DRAFTS.items():
        display(Markdown(f"---\n### {name}"))
        display(Markdown("**One-pass draft:**"))
        print(draft.strip())
        display(Markdown("**Polished rewrite:**"))
        polished = polish_kizomba_tutor_output(draft, processor, model, track_name=name)
        print(polished)
        if name in PER_TRACK:
            PER_TRACK[name]["polished"] = polished
else:
    print("RUN_POLISH=False; skipping polish pass")

## Collect outputs for review

Final cell — gathers every per-track output into a single Markdown dump and
writes it to `09_kizomba_extended_outputs.md` alongside the notebook. Useful
for reviewing the listening guide / tutor / polished output across the whole
batch without scraping the `.ipynb` (which carries large interactive
timelines per track).

In [ ]:
OUTPUT_PATH = Path("09_kizomba_extended_outputs.md")

_fence = chr(96) * 3
_source_summary = ", ".join(
    f"{source_label}={sum(1 for path in TRACKS if TRACK_SOURCE.get(path) == source_label)}"
    for source_label, _root in SOURCE_ROOTS
) or "none"
_parts = [
    f"# 09_kizomba batch outputs - model={CLOUD_MODEL_ID}\n",
    f"sources: {_source_summary}\n",
]
if RHYTHM_ANATOMY:
    _parts.append("## kizomba - rhythm_anatomy (genre intro)\n")
    _parts.append(f"{_fence}\n{RHYTHM_ANATOMY.strip()}\n{_fence}\n")
for name, data in PER_TRACK.items():
    _parts.append(f"## {name}\n")
    _parts.append(f"{_fence}\n{data['header']}\n{data['sections_text']}\n{_fence}\n")
    if data.get("listening"):
        _parts.append("### listening_guide\n")
        _parts.append(f"{_fence}\n{data['listening'].strip()}\n{_fence}\n")
    _parts.append("### kizomba_tutor (one-pass)\n")
    _parts.append(f"{_fence}\n{data['draft'].strip()}\n{_fence}\n")
    if data.get("drills"):
        _parts.append("### kizomba_drills (verified)\n")
        _parts.append(f"{_fence}\n{data['drills'].strip()}\n{_fence}\n")
    if data.get("drills_verified_stats"):
        _parts.append("### kizomba_drills verifier stats\n")
        _parts.append(f"{_fence}\n{data['drills_verified_stats'].strip()}\n{_fence}\n")
    drills_raw = data.get("drills_raw", "").strip()
    if drills_raw and drills_raw != data.get("drills", "").strip():
        _parts.append("### kizomba_drills raw Gemma draft\n")
        _parts.append(f"{_fence}\n{drills_raw}\n{_fence}\n")
    if data.get("transitions_table"):
        _parts.append("### describe_transitions\n")
        _parts.append(f"{_fence}\n{data['transitions_table'].strip()}\n{_fence}\n")
    if data.get("transitions"):
        _parts.append("### kizomba_transitions (verified)\n")
        _parts.append(f"{_fence}\n{data['transitions'].strip()}\n{_fence}\n")
    if data.get("transitions_verified_stats"):
        _parts.append("### kizomba_transitions verifier stats\n")
        _parts.append(f"{_fence}\n{data['transitions_verified_stats'].strip()}\n{_fence}\n")
    transitions_raw = data.get("transitions_raw", "").strip()
    if transitions_raw and transitions_raw != data.get("transitions", "").strip():
        _parts.append("### kizomba_transitions raw Gemma draft\n")
        _parts.append(f"{_fence}\n{transitions_raw}\n{_fence}\n")
    if data.get("polished"):
        _parts.append("### kizomba_tutor (polished)\n")
        _parts.append(f"{_fence}\n{data['polished'].strip()}\n{_fence}\n")
    # Phase 40d - unified timeline interleaves polished tutor (if available)
    # or one-pass draft with the verified transitions output.
    _tutor_text = data.get("polished", "").strip() or data.get("draft", "").strip()
    _transitions_text = data.get("transitions", "").strip()
    if _tutor_text and _transitions_text:
        _parts.append("### unified timeline (phases + transitions)\n")
        _parts.append(
            f"{_fence}\n{format_unified_timeline(_tutor_text, _transitions_text)}\n{_fence}\n"
        )

_body = "\n".join(_parts)
OUTPUT_PATH.write_text(_body)
print(_body)
print(f"\n[wrote {len(_body):,} chars across {len(PER_TRACK)} tracks to {OUTPUT_PATH.resolve()}]")